# Feature vs conjunction search: analysis

Treisman & Gelade (1980), Experiments I and II. For each condition and each answer
(target present = *positive*, absent = *negative*) this notebook computes the mean correct
RT at every display size, the linear regression of RT on display size (slope, intercept),
the percentage of the display-size variance that is linear, the ratio of positive to
negative slopes, and the error rates: the measures of the paper's Tables 1 and 2
(pp. 105, 109) and Figures 1 and 3.

**Input.** One run folder (`ANALYSIS_RUN`), several (`ANALYSIS_RUNS`, separated by the
OS path separator), or every `*_trials.csv` under `../data`. One run is one block in one
condition.

**Rules taken from the paper.** Dummy trials (the trial after each error) are dropped
(p. 103). RTs are from correct trials; an error's card was served again later, and its
correct repeat counts. Experiment I treated each subject's first block in each condition
as practice (p. 103): set `DROP_PRACTICE=1` to drop each subject's earliest run per
condition when they have more than one.

A simulated session is software verification, not data: its numbers come from the
generative model in `treisman_gelade1980/simulate.py`.

In [ ]:
import glob
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

HERE = Path.cwd()
if os.environ.get("ANALYSIS_RUN"):
    files = sorted(Path(os.environ["ANALYSIS_RUN"]).glob("*_trials.csv"))
elif os.environ.get("ANALYSIS_RUNS"):
    files = sorted(Path(f) for d in os.environ["ANALYSIS_RUNS"].split(os.pathsep)
                   for f in glob.glob(str(Path(d) / "**" / "*_trials.csv"), recursive=True))
else:
    files = sorted((HERE.parent / "data").rglob("*_trials.csv"))
assert files, "no *_trials.csv found: set ANALYSIS_RUN to a run folder"
DROP_PRACTICE = os.environ.get("DROP_PRACTICE", "0") == "1"


def read_run(path):
    df = pd.read_csv(path)
    m = re.search(r"sub-([^_/]+)_ses-(\d+)_run-(\d+)", path.name)
    df["subject"], df["session"], df["run"] = m.group(1), int(m.group(2)), int(m.group(3))
    df["run_file"] = path.name
    return df


trials = pd.concat([read_run(f) for f in files], ignore_index=True)
print(f"{len(files)} run(s), {len(trials)} rows; conditions: {sorted(trials['condition'].unique())}")

In [ ]:
# Which rows count.
trials["dummy"] = trials["dummy"].fillna(0).astype(int)
regular = trials[(trials["dummy"] == 0) & (trials["completed"].astype(bool))].copy()
if DROP_PRACTICE:
    order = regular.groupby(["subject", "condition"])["run_file"].transform(
        lambda s: s.map({f: i for i, f in enumerate(sorted(s.unique()))}))
    n_runs = regular.groupby(["subject", "condition"])["run_file"].transform("nunique")
    regular = regular[(order > 0) | (n_runs == 1)]
regular["answer"] = np.where(regular["target_present"].astype(int) == 1, "positive", "negative")
correct = regular[regular["outcome"] == "CORRECT"]
print(f"dummy trials dropped: {int((trials['dummy'] == 1).sum())}; "
      f"completed regular trials: {len(regular)}; correct: {len(correct)}")

In [ ]:
# Mean correct RT per subject, then across subjects (the paper reports group means).
per_subject = (correct.groupby(["condition", "answer", "display_size", "subject"])["rt_ms"]
               .mean().reset_index())
means = (per_subject.groupby(["condition", "answer", "display_size"])["rt_ms"]
         .agg(["mean", "count"]).reset_index())
means.pivot_table(index=["condition", "answer"], columns="display_size", values="mean").round(0)

In [ ]:
def fit(group):
    x = group["display_size"].to_numpy(float)
    y = group["mean"].to_numpy(float)
    if len(np.unique(x)) < 2:
        return pd.Series({"slope_ms_per_item": np.nan, "intercept_ms": np.nan, "pct_linear": np.nan})
    slope, intercept = np.polyfit(x, y, 1)
    pred = intercept + slope * x
    ss_between = ((y - y.mean()) ** 2).sum()
    pct = 100 * (1 - ((y - pred) ** 2).sum() / ss_between) if ss_between > 0 else np.nan
    return pd.Series({"slope_ms_per_item": slope, "intercept_ms": intercept, "pct_linear": pct})


regressions = means.groupby(["condition", "answer"]).apply(fit, include_groups=False).reset_index()
ratios = (regressions.pivot(index="condition", columns="answer", values="slope_ms_per_item")
          .assign(ratio_pos_to_neg=lambda d: d.get("positive") / d.get("negative")))
display(regressions.round(1))
display(ratios.round(2))

In [ ]:
# The paper's values (Table 1, p. 105; Table 2, p. 109), for comparison only.
paper = pd.DataFrame([
    ("conjunction", "positive", 28.7, 398), ("conjunction", "negative", 67.1, 397),
    ("feature", "positive", 3.1, 448), ("feature", "negative", 25.1, 514),
    ("difficult", "positive", 55.1, 453), ("difficult", "negative", 92.4, 472),
    ("easy", "positive", 20.5, 437), ("easy", "negative", 39.5, 489),
], columns=["condition", "answer", "paper_slope", "paper_intercept"])
regressions.merge(paper, on=["condition", "answer"], how="left").round(1)

In [ ]:
# Error rates: false negatives (misses on positive trials), false positives (on negative).
errors = (regular.assign(error=regular["outcome"] == "WRONG")
          .groupby(["condition", "answer", "display_size"])["error"].mean().mul(100)
          .unstack("display_size").round(1))
errors

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
styles = {"positive": "-", "negative": "--"}
for (cond, answer), g in means.groupby(["condition", "answer"]):
    ax.plot(g["display_size"], g["mean"], styles[answer], marker="o", label=f"{cond}, {answer}")
ax.set_xlabel("display size (items)")
ax.set_ylabel("mean correct RT (ms)")
ax.set_xticks(sorted(means["display_size"].unique()))
ax.legend(frameon=False, fontsize=8)
ax.set_title("Search time by display size")
fig.tight_layout()